# Analyse des décisions de transport — Expérience `current`

Ce notebook analyse les décisions de transport prises par les agents LLM à partir du fichier `moves.csv` de l'expérience **current**.

**Graphiques produits :**
- Répartition des modes proposés au LLM selon le lieu de résidence (3–4 sous-graphiques)
- Répartition des modes choisis selon l'origine de la décision (LLM vs cache sémantique)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../llm-agents/experiments/current/moves.csv")

df = pd.read_csv(DATA_PATH)
print(f"Trajets chargés : {len(df):,}")
print(f"Colonnes        : {df.columns.tolist()}")
df.head(3)

## Modes proposés au LLM selon le lieu de résidence

La colonne **`Modes proposés au LLM`** contient les itinéraires soumis au LLM, séparés par ` | `. Chaque occurrence est comptée individuellement par zone de résidence.

> Si la colonne est absente (ancienne version du logger), un avertissement est affiché.

In [ ]:
COL_MODES   = "Modes proposés au LLM"
COL_ZONE    = "Lieu de résidence"
COL_CHOISI  = "Mode de transport Choisi"
COL_REASON  = "Raisonnement"
CACHE_LABEL = "Décision récupérée depuis le cache sémantique LLM."

PALETTE = {
    "Voiture Privée":        "red",
    "Transports_collectifs": "green",
    "Vélo":                  "purple",
    "Marche":                "cyan",
    "Train":                 "purple",
    "Autres modes":          "#AAAAAA",
    "Aucun":                 "#DDDDDD",
}

ZONES = ["Toulouse", "1ere couronne", "2eme couronne"]

# ── Vérification de la colonne ───────────────────────────────────────────────
if COL_MODES not in df.columns:
    print(
        f"⚠  La colonne '{COL_MODES}' est absente de ce fichier moves.csv.\n"
        "   Elle est disponible à partir du logger v2 (expériences récentes).\n"
        "   → Remplacement par le mode choisi pour illustrer la structure."
    )
    df[COL_MODES] = df[COL_CHOISI]

# ── Explosion du champ multi-valeurs ────────────────────────────────────────
llm_rows = df[df[COL_MODES].notna() & (df[COL_MODES] != "")].copy()
exploded = (
    llm_rows[[COL_ZONE, COL_MODES]]
    .assign(**{COL_MODES: llm_rows[COL_MODES].str.split(" | ", regex=False)})
    .explode(COL_MODES)
)
exploded[COL_MODES] = exploded[COL_MODES].str.strip()

# ── Figure : 3 camemberts + 1 barres empilées ───────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle("Modes proposés au LLM selon le lieu de résidence", fontsize=15, fontweight="bold", y=1.01)

all_modes = sorted(exploded[COL_MODES].dropna().unique())

for ax, zone in zip(axes[:3], ZONES):
    subset   = exploded[exploded[COL_ZONE] == zone]
    counts   = subset[COL_MODES].value_counts().reindex(all_modes, fill_value=0)
    total    = counts.sum()
    non_zero = counts[counts > 0]
    cols_nz  = [PALETTE.get(m, "#CCCCCC") for m in non_zero.index]
    _, _, autotexts = ax.pie(
        non_zero,
        labels=None,
        colors=cols_nz,
        autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
        startangle=90,
        pctdistance=0.75,
        wedgeprops=dict(edgecolor="white", linewidth=1.2),
    )
    for at in autotexts:
        at.set_fontsize(9)
    ax.set_title(f"{zone}\n({total:,} occurrences)", fontsize=11, pad=8)

# Graphique 4 : barres empilées comparatives entre zones
ax_bar  = axes[3]
pivot   = (
    exploded.groupby([COL_ZONE, COL_MODES])
    .size()
    .unstack(fill_value=0)
    .reindex(ZONES)
    .reindex(columns=all_modes, fill_value=0)
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(ZONES))
x      = np.arange(len(ZONES))
for mode in all_modes:
    vals = pivot_pct[mode].values
    ax_bar.bar(x, vals, bottom=bottom, color=PALETTE.get(mode, "#CCCCCC"),
               label=mode, edgecolor="white", linewidth=0.8)
    bottom += vals

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(ZONES, fontsize=9)
ax_bar.set_ylabel("Part (%)")
ax_bar.set_ylim(0, 100)
ax_bar.set_title("Comparaison\npar zone", fontsize=11)
ax_bar.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))

# Légende commune
handles = [mpatches.Patch(color=PALETTE.get(m, "#CCCCCC"), label=m) for m in all_modes]
fig.legend(handles=handles, loc="lower center", ncol=len(all_modes),
           bbox_to_anchor=(0.5, -0.06), frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig("modes_proposes_par_zone.png", dpi=150, bbox_inches="tight")
plt.show()

## Mode de transport choisi : LLM direct vs cache sémantique

La colonne **`Raisonnement`** vaut exactement `"Décision récupérée depuis le cache sémantique LLM."` lorsque la décision a été récupérée depuis le cache — sinon une explication LLM ou un motif de fallback est présent.

In [ ]:
df["Source décision"] = df[COL_REASON].apply(
    lambda r: "Cache sémantique" if str(r).strip() == CACHE_LABEL else "LLM / autre"
)

n_cache = (df["Source décision"] == "Cache sémantique").sum()
n_total = len(df)
print(f"Décisions depuis le cache : {n_cache:,} / {n_total:,} ({100*n_cache/n_total:.1f}%)")
if n_cache == 0:
    print("ℹ  Aucune décision cache dans cette expérience — le cache n'était pas activé ou le seuil n'a pas été atteint.")

# ── Distribution des modes par source ───────────────────────────────────────
pivot_src = (
    df.groupby(["Source décision", COL_CHOISI])
    .size()
    .unstack(fill_value=0)
)
pivot_src_pct = pivot_src.div(pivot_src.sum(axis=1), axis=0) * 100

sources    = pivot_src_pct.index.tolist()
mode_cols  = pivot_src_pct.columns.tolist()
x          = np.arange(len(sources))
bar_width  = 0.15

fig, (ax_abs, ax_pct) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Mode de transport choisi — LLM direct vs cache sémantique",
    fontsize=13, fontweight="bold"
)

# --- Gauche : valeurs absolues ---
for i, mode in enumerate(mode_cols):
    abs_vals = pivot_src[mode].reindex(sources, fill_value=0).values
    ax_abs.bar(
        x + i * bar_width, abs_vals, bar_width,
        color=PALETTE.get(mode, "#CCCCCC"),
        label=mode, edgecolor="white", linewidth=0.6
    )
    for xi, v in zip(x + i * bar_width, abs_vals):
        if v > 0:
            ax_abs.text(xi, v + 2, str(int(v)), ha="center", va="bottom", fontsize=8)

ax_abs.set_xticks(x + bar_width * (len(mode_cols) - 1) / 2)
ax_abs.set_xticklabels(sources, fontsize=10)
ax_abs.set_ylabel("Nombre de trajets")
ax_abs.set_title("Effectifs")
ax_abs.legend(fontsize=9, frameon=False)

# --- Droite : pourcentages empilés ---
bottom = np.zeros(len(sources))
for mode in mode_cols:
    pct_vals = pivot_src_pct[mode].reindex(sources, fill_value=0).values
    bars = ax_pct.bar(
        sources, pct_vals, bottom=bottom,
        color=PALETTE.get(mode, "#CCCCCC"),
        label=mode, edgecolor="white", linewidth=0.8, width=0.45
    )
    for j, (bar, pct) in enumerate(zip(bars, pct_vals)):
        if pct > 4:
            ax_pct.text(
                bar.get_x() + bar.get_width() / 2,
                bottom[j] + pct / 2,
                f"{pct:.1f}%",
                ha="center", va="center", fontsize=9, color="white", fontweight="bold"
            )
    bottom += pct_vals

ax_pct.set_ylim(0, 100)
ax_pct.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax_pct.set_ylabel("Part (%)")
ax_pct.set_title("Répartition relative")

handles = [mpatches.Patch(color=PALETTE.get(m, "#CCCCCC"), label=m) for m in mode_cols]
fig.legend(handles=handles, loc="lower center", ncol=len(mode_cols),
           bbox_to_anchor=(0.5, -0.08), frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig("mode_choisi_cache_vs_llm.png", dpi=150, bbox_inches="tight")
plt.show()